In [1]:
# Cell 1 — Setup, imports, Google Drive, and LightGBM
from __future__ import annotations

import os, sys, gc, json, math, time, random, hashlib, shutil, zipfile, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, roc_auc_score, roc_curve,
)

try:
    import lightgbm as lgb
except ImportError:
    import subprocess
    print('Installing LightGBM...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm>=4.3,<5'])
    import lightgbm as lgb

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Không chạy trong Google Colab; bỏ qua bước mount Drive.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | PyTorch: {torch.__version__} | LightGBM: {lgb.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB')

Mounted at /content/drive
Device: cuda | PyTorch: 2.11.0+cu128 | LightGBM: 4.6.0
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB


In [2]:
# Cell 2 — Step 7 configuration

STEP5_EXPERIMENT_TAG = 'conference_step5_literature_subset_v1'
STEP5_PHASE = 'confirmation'
STEP7_EXPERIMENT_TAG = 'conference_step7_tabular_loo_v1'
PROTOCOL_NAME = 'exact_feature_group'
BENCHMARK_SPLIT_SEED = 2027
MODEL_SEEDS = [42, 123, 456]

DRIVE_RUN_ROOT = Path('/content/drive/MyDrive/THAT_IDS_RERUN')
STEP5_ROOT = DRIVE_RUN_ROOT / STEP5_EXPERIMENT_TAG
STEP5_CONFIRMATION_ROOT = STEP5_ROOT / STEP5_PHASE
STEP5_SUBSET_DEFINITION_ROOT = STEP5_ROOT / 'subset_definition'
OUTPUT_ROOT = DRIVE_RUN_ROOT / STEP7_EXPERIMENT_TAG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = {
    'UNSW_V2': [
        '/content/drive/MyDrive/datasets/NF-UNSW-NB15-v2.csv',
        '/content/drive/MyDrive/NF-UNSW-NB15-v2.csv',
    ],
    'CIC_V2': [
        '/content/drive/MyDrive/datasets/NF-CSE-CIC-IDS2018-v2.csv',
        '/content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv',
    ],
}

# Có thể chạy một dataset mỗi lần để giảm rủi ro mất runtime.
DATASETS_TO_RUN = ['CIC_V2', 'UNSW_V2']
RUN_LIGHTGBM = True
RUN_FULL_LOO = True
STRICT_COMPLETENESS = False

CSV_CHUNK_SIZE = 500_000
GRAPH_WINDOW_SIZE = 5000
CLIP_MIN, CLIP_MAX = -10.0, 10.0

# Cross-fitted leave-target-out control.
LOO_FOLDS = 5
LOO_FOLD_SEED = 2027
LOO_ZERO_UNSEEN_NODES = True

# Exact Step-5 Full architecture constants.
HIDDEN_DIM = 96
AE_HIDDEN = 192
LATENT_DIM = 96
GRAPH_DROPOUT = 0.10
ENC_DROPOUT = 0.10
CLS_DROPOUT = 0.15
INIT_GATE_LOGIT = -1.5
CTX_RECON_WEIGHT = 0.10
SCORE_CLS_WEIGHT = 0.90
SCORE_RECON_WEIGHT = 0.10

# Fixed LightGBM configuration — same for both datasets; no post-hoc tuning.
LIGHTGBM_CONFIG = {
    'objective': 'binary',
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 50,
    'subsample': 0.80,
    'subsample_freq': 1,
    'colsample_bytree': 0.80,
    'reg_alpha': 0.0,
    'reg_lambda': 1.0,
    'max_bin': 255,
    'n_jobs': -1,
    'verbosity': -1,
    'deterministic': True,
    'force_col_wise': True,
}
LIGHTGBM_EARLY_STOPPING_ROUNDS = 100

TEST_CURVE_FPR_TARGETS = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
TEST_CURVE_RECALL_TARGETS = [0.90, 0.95, 0.99]

print('Step 5 confirmation root:', STEP5_CONFIRMATION_ROOT)
print('Step 7 output root:', OUTPUT_ROOT)
print('Datasets:', DATASETS_TO_RUN)
print('Model seeds:', MODEL_SEEDS)
print('LOO folds:', LOO_FOLDS)

Step 5 confirmation root: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step5_literature_subset_v1/confirmation
Step 7 output root: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step7_tabular_loo_v1
Datasets: ['CIC_V2', 'UNSW_V2']
Model seeds: [42, 123, 456]
LOO folds: 5


In [3]:
# Cell 3 — Reproducibility, paths, checkpoints, and JSON helpers

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def load_json(path: Path) -> Dict[str, Any]:
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_torch_checkpoint(path: Path, map_location: torch.device) -> Dict[str, Any]:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def resolve_dataset_path(dataset_name: str) -> str:
    for candidate in DATA_PATH_CANDIDATES.get(dataset_name, []):
        if Path(candidate).exists():
            return candidate
    raise FileNotFoundError(
        'Không tìm thấy dataset ' + dataset_name + '. Đã thử:\n' +
        '\n'.join(DATA_PATH_CANDIDATES.get(dataset_name, []))
    )


def step5_dataset_dir(dataset_name: str) -> Path:
    return STEP5_CONFIRMATION_ROOT / dataset_name


def step5_shared_dir(dataset_name: str) -> Path:
    return step5_dataset_dir(dataset_name) / PROTOCOL_NAME / 'shared' / f'seed_{BENCHMARK_SPLIT_SEED}'


def subset_definition_dir(dataset_name: str) -> Path:
    return STEP5_SUBSET_DEFINITION_ROOT / dataset_name


def step5_full_task_dir(dataset_name: str, seed: int) -> Path:
    return step5_dataset_dir(dataset_name) / PROTOCOL_NAME / 'full' / f'seed_{seed}'


def step7_task_dir(dataset_name: str, model_name: str, seed: int) -> Path:
    return OUTPUT_ROOT / dataset_name / PROTOCOL_NAME / model_name / f'seed_{seed}'


def config_fingerprint(payload: Dict[str, Any]) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode('utf-8')).hexdigest()[:20]


def preflight_step5(dataset_name: str) -> None:
    required = [
        subset_definition_dir(dataset_name) / 'subset_indices_int32.npy',
        subset_definition_dir(dataset_name) / 'subset_manifest.json',
        step5_dataset_dir(dataset_name) / 'model_feature_names.csv',
        step5_shared_dir(dataset_name) / 'split_indices.npz',
        step5_shared_dir(dataset_name) / 'split_metadata.json',
        step5_shared_dir(dataset_name) / 'split_overlap_audit.json',
        step5_shared_dir(dataset_name) / 'scaler.joblib',
    ]
    if RUN_FULL_LOO:
        for seed in MODEL_SEEDS:
            required += [
                step5_full_task_dir(dataset_name, seed) / 'best.pt',
                step5_full_task_dir(dataset_name, seed) / 'test_metrics.json',
                step5_full_task_dir(dataset_name, seed) / 'test_predictions.npz',
            ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError('Thiếu Step-5 artifacts:\n' + '\n'.join(missing))
    print(f'Preflight OK: {dataset_name}')

In [4]:
# Cell 4 — Load the exact frozen Step-5 subset, split, and scaler

@dataclass
class FixedSubsetBundle:
    name: str
    feature_names: List[str]
    X_raw: np.ndarray
    src_ip: np.ndarray
    dst_ip: np.ndarray
    protocol: np.ndarray
    labels: np.ndarray
    attack_codes: np.ndarray
    attack_classes: np.ndarray
    subset_global_indices: np.ndarray
    subset_fingerprint: str


def read_step5_feature_names(dataset_name: str) -> List[str]:
    path = step5_dataset_dir(dataset_name) / 'model_feature_names.csv'
    table = pd.read_csv(path)
    return table.sort_values('feature_index')['feature'].astype(str).tolist()


def load_fixed_subset_bundle(dataset_name: str) -> FixedSubsetBundle:
    csv_path = Path(resolve_dataset_path(dataset_name))
    subset_idx = np.asarray(
        np.load(subset_definition_dir(dataset_name) / 'subset_indices_int32.npy', mmap_mode='r'),
        dtype=np.int64,
    )
    manifest = load_json(subset_definition_dir(dataset_name) / 'subset_manifest.json')
    if np.any(np.diff(subset_idx) < 0):
        raise RuntimeError('Step-5 subset indices are not sorted.')

    feature_names = read_step5_feature_names(dataset_name)
    context_cols = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'PROTOCOL', 'Label', 'Attack']
    usecols = list(dict.fromkeys(feature_names + context_cols))

    print(f'\nLoading frozen subset {dataset_name}: {len(subset_idx):,} rows, {len(feature_names)} features')
    feature_chunks: List[np.ndarray] = []
    context_chunks: Dict[str, List[np.ndarray]] = {c: [] for c in context_cols}
    cursor = 0
    global_start = 0
    t0 = time.time()

    for chunk_no, df_chunk in enumerate(
        pd.read_csv(csv_path, usecols=usecols, chunksize=CSV_CHUNK_SIZE, low_memory=False), start=1
    ):
        global_end = global_start + len(df_chunk)
        left = np.searchsorted(subset_idx, global_start, side='left')
        right = np.searchsorted(subset_idx, global_end, side='left')
        if right > left:
            local_rows = subset_idx[left:right] - global_start
            selected = df_chunk.iloc[local_rows]
            x = np.empty((len(selected), len(feature_names)), dtype=np.float32)
            for j, col in enumerate(feature_names):
                arr = pd.to_numeric(selected[col], errors='coerce').to_numpy(dtype=np.float32, copy=False)
                x[:, j] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            feature_chunks.append(x)
            for col in context_cols:
                context_chunks[col].append(selected[col].to_numpy(copy=True))
            cursor += len(selected)
        global_start = global_end
        if chunk_no % 10 == 0 or cursor == len(subset_idx):
            print(f'  chunks={chunk_no} | selected={cursor:,}/{len(subset_idx):,}')
        del df_chunk
        if cursor == len(subset_idx):
            break

    if cursor != len(subset_idx):
        raise RuntimeError(f'Loaded {cursor:,} rows; expected {len(subset_idx):,}.')

    X_raw = np.concatenate(feature_chunks, axis=0)
    context = {k: np.concatenate(v, axis=0) for k, v in context_chunks.items()}
    labels = pd.to_numeric(pd.Series(context['Label']), errors='coerce').fillna(0).to_numpy(dtype=np.int8)
    attack_codes, attack_classes = pd.factorize(
        pd.Series(context['Attack']).fillna('<MISSING_ATTACK>').astype(str), sort=False
    )
    if not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError('Label is not binary.')

    print(f'Loaded in {(time.time()-t0)/60:.1f} min | label counts:', dict(zip(*np.unique(labels, return_counts=True))))
    return FixedSubsetBundle(
        name=dataset_name,
        feature_names=feature_names,
        X_raw=X_raw,
        src_ip=context['IPV4_SRC_ADDR'],
        dst_ip=context['IPV4_DST_ADDR'],
        protocol=context['PROTOCOL'],
        labels=labels,
        attack_codes=attack_codes.astype(np.int16, copy=False),
        attack_classes=np.asarray(attack_classes, dtype=str),
        subset_global_indices=subset_idx,
        subset_fingerprint=str(manifest['subset_fingerprint']),
    )


def load_frozen_step5_artifacts(bundle: FixedSubsetBundle):
    shared = step5_shared_dir(bundle.name)
    split_data = np.load(shared / 'split_indices.npz')
    train_idx = np.asarray(split_data['train_idx'], dtype=np.int64)
    val_idx = np.asarray(split_data['val_idx'], dtype=np.int64)
    test_idx = np.asarray(split_data['test_idx'], dtype=np.int64)
    split_meta = load_json(shared / 'split_metadata.json')
    overlap = load_json(shared / 'split_overlap_audit.json')
    scaler = joblib.load(shared / 'scaler.joblib')

    if split_meta.get('protocol_name') != PROTOCOL_NAME:
        raise RuntimeError('Step-5 protocol mismatch.')
    if max(float(overlap.get('validation_feature_overlap_rate', 1)), float(overlap.get('test_feature_overlap_rate', 1))) != 0.0:
        raise RuntimeError('Frozen Step-5 split has non-zero exact-feature overlap.')
    all_idx = np.concatenate([train_idx, val_idx, test_idx])
    if len(np.unique(all_idx)) != len(all_idx):
        raise RuntimeError('Frozen split row indices overlap.')

    print(f"Frozen split: train={len(train_idx):,}, val={len(val_idx):,}, test={len(test_idx):,} | {split_meta['split_fingerprint']}")
    return train_idx, val_idx, test_idx, scaler, split_meta

In [5]:
# Cell 5 — Shared metrics, validation threshold, per-attack recall, and curve points

def safe_divide(a: float, b: float) -> float:
    return float(a / b) if b else 0.0


def find_best_threshold_macro_f1_exact(scores: np.ndarray, labels: np.ndarray) -> Tuple[float, float]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int8)
    order = np.argsort(-scores, kind='stable')
    s = scores[order]
    y = labels[order]
    total_pos = int(y.sum())
    total_neg = len(y) - total_pos
    cum_tp = np.cumsum(y)
    cum_fp = np.cumsum(1-y)
    ends = np.flatnonzero(np.r_[s[:-1] != s[1:], True])
    tp = cum_tp[ends].astype(float); fp = cum_fp[ends].astype(float)
    fn = total_pos - tp; tn = total_neg - fp
    f1a = np.divide(2*tp, 2*tp+fp+fn, out=np.zeros_like(tp), where=(2*tp+fp+fn)>0)
    f1b = np.divide(2*tn, 2*tn+fp+fn, out=np.zeros_like(tn), where=(2*tn+fp+fn)>0)
    macro = (f1a+f1b)/2
    best = int(np.argmax(macro))
    return float(s[ends[best]]), float(macro[best])


def orient_validation_scores(scores: np.ndarray, labels: np.ndarray) -> Tuple[np.ndarray, int]:
    sign = -1 if roc_auc_score(labels, scores) < 0.5 else 1
    return sign * np.asarray(scores, dtype=np.float64), sign


def evaluate_binary_scores(scores: np.ndarray, labels: np.ndarray, threshold: float) -> Dict[str, Any]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int8)
    pred = (scores > float(threshold)).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0,1]).ravel()
    return {
        'threshold': float(threshold),
        'f1_macro': float(f1_score(labels, pred, average='macro', zero_division=0)),
        'auc_roc': float(roc_auc_score(labels, scores)),
        'pr_auc': float(average_precision_score(labels, scores)),
        'accuracy': safe_divide(tp+tn, tp+tn+fp+fn),
        'precision': safe_divide(tp, tp+fp),
        'recall': safe_divide(tp, tp+fn),
        'specificity': safe_divide(tn, tn+fp),
        'balanced_accuracy': 0.5*(safe_divide(tp,tp+fn)+safe_divide(tn,tn+fp)),
        'f1_attack': safe_divide(2*tp, 2*tp+fp+fn),
        'f1_benign': safe_divide(2*tn, 2*tn+fp+fn),
        'fp_rate': safe_divide(fp, fp+tn),
        'tp_rate': safe_divide(tp, tp+fn),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'predictions': pred,
    }


def per_attack_recall(pred: np.ndarray, labels: np.ndarray, attack_codes: np.ndarray, attack_classes: np.ndarray):
    rows=[]
    for code, name in enumerate(attack_classes):
        mask=(attack_codes==code)&(labels==1)
        n=int(mask.sum())
        if n:
            detected=int(pred[mask].sum())
            rows.append({'attack':str(name),'count':n,'detected':detected,'recall':detected/n})
    return rows


def test_curve_operating_points(scores: np.ndarray, labels: np.ndarray):
    fpr,tpr,rt=roc_curve(labels,scores)
    precision,recall,pt=precision_recall_curve(labels,scores)
    rows=[]
    for target in TEST_CURVE_FPR_TARGETS:
        eligible=np.flatnonzero(fpr<=target+1e-15)
        chosen=eligible[np.argmax(tpr[eligible])] if len(eligible) else 0
        rows.append({'criterion':'TPR_at_test_FPR','target':target,'value':float(tpr[chosen]),'observed':float(fpr[chosen]),'curve_threshold':float(rt[chosen])})
    for target in TEST_CURVE_RECALL_TARGETS:
        eligible=np.flatnonzero(recall>=target-1e-15)
        chosen=eligible[np.argmax(precision[eligible])] if len(eligible) else int(np.argmax(recall))
        thr=float('nan') if chosen>=len(pt) else float(pt[chosen])
        rows.append({'criterion':'Precision_at_test_recall','target':target,'value':float(precision[chosen]),'observed':float(recall[chosen]),'curve_threshold':thr})
    return rows

In [6]:
# Cell 6 — LightGBM baseline on the frozen Step-5 rows/split/scaler

def run_lightgbm_task(bundle, train_idx, val_idx, test_idx, scaler, split_meta, seed: int):
    model_name='lightgbm'
    task_dir=step7_task_dir(bundle.name,model_name,seed)
    task_dir.mkdir(parents=True,exist_ok=True)
    result_path=task_dir/'test_metrics.json'
    pred_path=task_dir/'test_predictions.npz'
    complete_path=task_dir/'completed.json'

    y_train=bundle.labels[train_idx]
    scale_pos_weight=min(float((y_train==0).sum()/max((y_train==1).sum(),1)),12.0)
    cfg={**LIGHTGBM_CONFIG,
         'random_state':seed,'bagging_seed':seed,'feature_fraction_seed':seed,'data_random_seed':seed,
         'scale_pos_weight':scale_pos_weight}
    fp=config_fingerprint({'step7':STEP7_EXPERIMENT_TAG,'dataset':bundle.name,'model':model_name,'seed':seed,
                           'subset':bundle.subset_fingerprint,'split':split_meta['split_fingerprint'],'features':bundle.feature_names,'config':cfg})
    if complete_path.exists() and result_path.exists() and pred_path.exists():
        done=load_json(complete_path)
        if done.get('config_fingerprint')==fp:
            print(f'  SKIP LightGBM {bundle.name} seed={seed}')
            return load_json(result_path)
        raise RuntimeError(f'Config mismatch: {task_dir}')

    seed_everything(seed)
    t0=time.time()
    # Reuse the train-fitted Step-5 scaler to keep the input protocol identical.
    X_train=np.clip(scaler.transform(bundle.X_raw[train_idx]),CLIP_MIN,CLIP_MAX).astype(np.float32)
    X_val=np.clip(scaler.transform(bundle.X_raw[val_idx]),CLIP_MIN,CLIP_MAX).astype(np.float32)
    X_test=np.clip(scaler.transform(bundle.X_raw[test_idx]),CLIP_MIN,CLIP_MAX).astype(np.float32)
    y_val=bundle.labels[val_idx]; y_test=bundle.labels[test_idx]

    model=lgb.LGBMClassifier(**cfg)
    model.fit(
        X_train,y_train,
        eval_set=[(X_val,y_val)],
        eval_metric='binary_logloss',
        callbacks=[lgb.early_stopping(LIGHTGBM_EARLY_STOPPING_ROUNDS,verbose=False), lgb.log_evaluation(period=50)],
        feature_name=bundle.feature_names,
    )
    best_iteration=int(model.best_iteration_ or cfg['n_estimators'])
    val_scores=model.predict_proba(X_val,num_iteration=best_iteration)[:,1]
    test_scores=model.predict_proba(X_test,num_iteration=best_iteration)[:,1]
    val_scores,score_sign=orient_validation_scores(val_scores,y_val)
    test_scores=score_sign*test_scores
    threshold,val_f1=find_best_threshold_macro_f1_exact(val_scores,y_val)
    test_eval=evaluate_binary_scores(test_scores,y_test,threshold)
    attacks=per_attack_recall(test_eval['predictions'],y_test,bundle.attack_codes[test_idx],bundle.attack_classes)

    model.booster_.save_model(str(task_dir/'model.txt'),num_iteration=best_iteration)
    pd.DataFrame({'feature':bundle.feature_names,
                  'gain':model.booster_.feature_importance(importance_type='gain'),
                  'split':model.booster_.feature_importance(importance_type='split')}).sort_values('gain',ascending=False).to_csv(task_dir/'feature_importance.csv',index=False)

    result={'dataset':bundle.name,'protocol':PROTOCOL_NAME,'model':model_name,'seed':seed,
            'best_iteration':best_iteration,'elapsed_seconds':time.time()-t0,
            'validation':{'f1_macro':val_f1,'threshold':threshold,'score_sign':score_sign},
            'test':{k:v for k,v in test_eval.items() if k!='predictions'},'per_attack':attacks,
            'subset_fingerprint':bundle.subset_fingerprint,'split_fingerprint':split_meta['split_fingerprint'],
            'feature_names':bundle.feature_names,'config':cfg,'config_fingerprint':fp}
    atomic_json_dump(result,result_path)
    np.savez_compressed(pred_path,scores=test_scores.astype(np.float32),predictions=test_eval['predictions'].astype(np.int8),
                        labels=y_test.astype(np.int8),attack_codes=bundle.attack_codes[test_idx].astype(np.int16),
                        attack_classes=bundle.attack_classes.astype(str),threshold=np.asarray([threshold]))
    atomic_json_dump({'completed':True,'config_fingerprint':fp,'completed_at_unix':time.time()},complete_path)
    print(f"  DONE LightGBM seed={seed} | F1={test_eval['f1_macro']:.4f} | PR={test_eval['pr_auc']:.4f} | FPR={100*test_eval['fp_rate']:.4f}%")
    del X_train,X_val,X_test,model
    gc.collect()
    return result

In [7]:
# Cell 7 — Exact Step-5 Full architecture for checkpoint loading

@dataclass(frozen=True)
class VariantConfig:
    name: str; graph_mode: str; edge_control: str
    use_decoder: bool; use_classifier: bool
    train_reconstruction: bool; train_classifier: bool; inference_mode: str

FULL_VARIANT=VariantConfig('full','hetero','observed',True,True,True,True,'hybrid')

class RelationAttentionEncoder(nn.Module):
    def __init__(self,in_dim:int,out_dim:int,n_relations:int,dropout:float=0.10):
        super().__init__(); self.out_dim=out_dim; self.n_relations=n_relations
        self.proj=nn.Linear(in_dim,out_dim)
        self.edge_attention=nn.ModuleList([nn.Linear(out_dim*2,1) for _ in range(n_relations)])
        self.semantic_attention=nn.Linear(out_dim,1)
        self.dropout=nn.Dropout(dropout)
        self.norms=nn.ModuleList([nn.LayerNorm(out_dim) for _ in range(n_relations)])
    def aggregate_relation(self,h,edge_src,edge_dst,relation_idx):
        if edge_src.numel()==0: return h
        src_h=h[edge_src]; dst_h=h[edge_dst]
        alpha=torch.sigmoid(self.edge_attention[relation_idx](torch.cat([src_h,dst_h],dim=1)))
        messages=src_h*alpha
        out=torch.zeros_like(h); weight_sum=torch.zeros(h.size(0),1,device=h.device,dtype=h.dtype)
        out.scatter_add_(0,edge_dst.unsqueeze(1).expand(-1,self.out_dim),messages)
        weight_sum.scatter_add_(0,edge_dst.unsqueeze(1),alpha)
        out=out/weight_sum.clamp(min=1.0)
        return self.norms[relation_idx](h+self.dropout(out))
    def forward(self,node_features,relations):
        h=F.relu(self.proj(node_features))
        outputs=[self.aggregate_relation(h,s,d,i) for i,(s,d) in enumerate(relations)]
        stacked=torch.stack(outputs,dim=0)
        weights=torch.softmax(self.semantic_attention(stacked),dim=0)
        return (weights*stacked).sum(dim=0)

class HybridDetector(nn.Module):
    def __init__(self,input_dim,flow_dim,hidden_dim,latent_dim,use_decoder,use_classifier,ctx_recon_weight):
        super().__init__(); self.flow_dim=flow_dim; self.ctx_recon_weight=ctx_recon_weight
        self.encoder=nn.Sequential(nn.Linear(input_dim,hidden_dim),nn.ReLU(),nn.Dropout(ENC_DROPOUT),nn.Linear(hidden_dim,latent_dim),nn.ReLU())
        self.decoder=nn.Sequential(nn.Linear(latent_dim,hidden_dim),nn.ReLU(),nn.Linear(hidden_dim,input_dim)) if use_decoder else None
        self.classifier=nn.Sequential(nn.Linear(input_dim+latent_dim,hidden_dim),nn.ReLU(),nn.Dropout(CLS_DROPOUT),
                                      nn.Linear(hidden_dim,hidden_dim//2),nn.ReLU(),nn.Linear(hidden_dim//2,1)) if use_classifier else None
    def forward(self,x):
        z=self.encoder(x); reconstruction=self.decoder(z) if self.decoder is not None else None
        logits=self.classifier(torch.cat([x,z],dim=1)).squeeze(1) if self.classifier is not None else None
        return z,reconstruction,logits
    def reconstruction_error(self,x,reconstruction):
        err=(reconstruction-x)**2
        flow_err=err[:,:self.flow_dim].mean(dim=1); context_err=err[:,self.flow_dim:].mean(dim=1)
        return flow_err+self.ctx_recon_weight*context_err

class THATIDSVariantModel(nn.Module):
    def __init__(self,flow_dim:int,variant:VariantConfig):
        super().__init__(); self.flow_dim=flow_dim; self.variant=variant
        self.graph_encoder=RelationAttentionEncoder(flow_dim,HIDDEN_DIM,2,GRAPH_DROPOUT)
        self.context_gate=nn.Parameter(torch.tensor(float(INIT_GATE_LOGIT)))
        self.detector=HybridDetector(flow_dim+HIDDEN_DIM,flow_dim,AE_HIDDEN,LATENT_DIM,True,True,CTX_RECON_WEIGHT)
    def gate_value(self): return float(torch.sigmoid(self.context_gate.detach()).item())


def load_step5_full_model(dataset_name: str, seed: int, flow_dim: int):
    checkpoint_path=step5_full_task_dir(dataset_name,seed)/'best.pt'
    checkpoint=load_torch_checkpoint(checkpoint_path,DEVICE)
    model=THATIDSVariantModel(flow_dim,FULL_VARIANT).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'],strict=True)
    model.eval()
    return model,checkpoint

In [8]:
# Cell 8 — Cross-fitted leave-target-out graph windows and inference

@dataclass
class LOOGraphWindow:
    edge_feat: torch.Tensor
    labels: torch.Tensor
    attack_codes: torch.Tensor
    src: torch.Tensor
    dst: torch.Tensor
    proto_local: torch.Tensor
    fold_ids: torch.Tensor
    n_ip: int
    n_proto: int


def _str_array(values,missing):
    return pd.Series(values,copy=False).fillna(missing).astype(str).to_numpy()


def build_loo_window(bundle,row_idx,scaler,window_no:int):
    scaled=np.clip(scaler.transform(bundle.X_raw[row_idx]),CLIP_MIN,CLIP_MAX).astype(np.float32,copy=False)
    srcv=_str_array(bundle.src_ip[row_idx],'<MISSING_IP>'); dstv=_str_array(bundle.dst_ip[row_idx],'<MISSING_IP>')
    pv=_str_array(bundle.protocol[row_idx],'<MISSING_PROTOCOL>')
    codes,_=pd.factorize(np.concatenate([srcv,dstv]),sort=False); n=len(row_idx)
    src=codes[:n].astype(np.int64); dst=codes[n:].astype(np.int64)
    proto,pu=pd.factorize(pv,sort=False); proto=proto.astype(np.int64)
    rng=np.random.default_rng(LOO_FOLD_SEED+1_000_003*window_no)
    order=rng.permutation(n); folds=np.empty(n,dtype=np.int64); folds[order]=np.arange(n)%LOO_FOLDS
    return LOOGraphWindow(torch.from_numpy(scaled),torch.from_numpy(bundle.labels[row_idx].astype(np.int64)),
                          torch.from_numpy(bundle.attack_codes[row_idx].astype(np.int64)),torch.from_numpy(src),torch.from_numpy(dst),
                          torch.from_numpy(proto),torch.from_numpy(folds),int(codes.max()+1),int(len(pu)))


def prepare_loo_windows(bundle,split_idx,scaler,description):
    windows=[]; nwin=math.ceil(len(split_idx)/GRAPH_WINDOW_SIZE); t0=time.time()
    for w in range(nwin):
        idx=split_idx[w*GRAPH_WINDOW_SIZE:(w+1)*GRAPH_WINDOW_SIZE]
        if len(idx): windows.append(build_loo_window(bundle,idx,scaler,w))
        if (w+1)%100==0 or w+1==nwin: print(f'  {description}: {w+1}/{nwin}')
    print(f'Prepared {len(windows)} {description} windows in {(time.time()-t0)/60:.1f} min')
    return windows


@torch.no_grad()
def collect_crossfit_loo_scores(model,windows):
    model.eval(); all_scores=[]; all_labels=[]; all_attacks=[]
    for wi,window in enumerate(windows):
        feat=window.edge_feat.to(DEVICE); labels=window.labels.to(DEVICE); attacks=window.attack_codes.to(DEVICE)
        src=window.src.to(DEVICE); dst=window.dst.to(DEVICE); proto=window.proto_local.to(DEVICE)+window.n_ip
        folds=window.fold_ids.to(DEVICE); n_nodes=window.n_ip+window.n_proto
        for fold in range(LOO_FOLDS):
            query=folds==fold; context=~query
            if not query.any(): continue
            cf=feat[context]; cs=src[context]; cd=dst[context]; cp=proto[context]
            node_feat=torch.zeros(n_nodes,feat.size(1),device=DEVICE,dtype=feat.dtype)
            counts=torch.zeros(n_nodes,1,device=DEVICE,dtype=feat.dtype)
            ones=torch.ones(cf.size(0),1,device=DEVICE,dtype=feat.dtype)
            for ids in (cs,cd,cp):
                node_feat.scatter_add_(0,ids.unsqueeze(1).expand(-1,feat.size(1)),cf)
                counts.scatter_add_(0,ids.unsqueeze(1),ones)
            node_feat=node_feat/counts.clamp(min=1.0)
            comm_src=torch.cat([cs,cd]); comm_dst=torch.cat([cd,cs])
            proto_src=torch.cat([cs,cp]); proto_dst=torch.cat([cp,cs])
            emb=model.graph_encoder(node_feat,[(comm_src,comm_dst),(proto_src,proto_dst)])
            if LOO_ZERO_UNSEEN_NODES:
                emb=emb*(counts>0).to(emb.dtype)
            qsrc=src[query]; qdst=dst[query]; qproto=proto[query]
            context_vec=torch.sigmoid(model.context_gate)*(emb[qsrc]+emb[qdst]+emb[qproto])/3.0
            detector_input=torch.cat([feat[query],context_vec],dim=1)
            _,reconstruction,logits=model.detector(detector_input)
            probability=torch.sigmoid(logits)
            rec_error=model.detector.reconstruction_error(detector_input,reconstruction)
            score=SCORE_CLS_WEIGHT*probability+SCORE_RECON_WEIGHT*torch.log1p(rec_error)
            all_scores.append(score.cpu().numpy().astype(np.float32))
            all_labels.append(labels[query].cpu().numpy().astype(np.int8))
            all_attacks.append(attacks[query].cpu().numpy().astype(np.int16))
        if (wi+1)%100==0 or wi+1==len(windows): print(f'    inferred {wi+1}/{len(windows)} windows')
    return {'scores':np.concatenate(all_scores),'labels':np.concatenate(all_labels),'attack_codes':np.concatenate(all_attacks)}


def run_full_loo_task(bundle,val_windows,test_windows,split_meta,seed:int):
    model_name='full_loo_crossfit'
    task_dir=step7_task_dir(bundle.name,model_name,seed); task_dir.mkdir(parents=True,exist_ok=True)
    result_path=task_dir/'test_metrics.json'; pred_path=task_dir/'test_predictions.npz'; complete_path=task_dir/'completed.json'
    cfg={'loo_folds':LOO_FOLDS,'fold_seed':LOO_FOLD_SEED,'zero_unseen_nodes':LOO_ZERO_UNSEEN_NODES,
         'inference_checkpoint':'Step5 Full best.pt','retraining':False,
         'definition':'Each target fold is predicted from a graph built only from the other folds.'}
    fp=config_fingerprint({'step7':STEP7_EXPERIMENT_TAG,'dataset':bundle.name,'model':model_name,'seed':seed,
                           'subset':bundle.subset_fingerprint,'split':split_meta['split_fingerprint'],'features':bundle.feature_names,'config':cfg})
    if complete_path.exists() and result_path.exists() and pred_path.exists():
        done=load_json(complete_path)
        if done.get('config_fingerprint')==fp:
            print(f'  SKIP Full-LOO {bundle.name} seed={seed}')
            return load_json(result_path)
        raise RuntimeError(f'Config mismatch: {task_dir}')
    seed_everything(seed); t0=time.time()
    model,checkpoint=load_step5_full_model(bundle.name,seed,len(bundle.feature_names))
    val=collect_crossfit_loo_scores(model,val_windows); test=collect_crossfit_loo_scores(model,test_windows)
    val_scores,score_sign=orient_validation_scores(val['scores'],val['labels']); test_scores=score_sign*test['scores']
    threshold,val_f1=find_best_threshold_macro_f1_exact(val_scores,val['labels'])
    test_eval=evaluate_binary_scores(test_scores,test['labels'],threshold)
    attacks=per_attack_recall(test_eval['predictions'],test['labels'],test['attack_codes'],bundle.attack_classes)
    result={'dataset':bundle.name,'protocol':PROTOCOL_NAME,'model':model_name,'seed':seed,
            'source_full_best_epoch':int(checkpoint.get('epoch',-1)),'parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
            'gate':model.gate_value(),'elapsed_seconds':time.time()-t0,
            'validation':{'f1_macro':val_f1,'threshold':threshold,'score_sign':score_sign},
            'test':{k:v for k,v in test_eval.items() if k!='predictions'},'per_attack':attacks,
            'subset_fingerprint':bundle.subset_fingerprint,'split_fingerprint':split_meta['split_fingerprint'],
            'config':cfg,'config_fingerprint':fp}
    atomic_json_dump(result,result_path)
    np.savez_compressed(pred_path,scores=test_scores.astype(np.float32),predictions=test_eval['predictions'].astype(np.int8),
                        labels=test['labels'].astype(np.int8),attack_codes=test['attack_codes'].astype(np.int16),
                        attack_classes=bundle.attack_classes.astype(str),threshold=np.asarray([threshold]))
    atomic_json_dump({'completed':True,'config_fingerprint':fp,'completed_at_unix':time.time()},complete_path)
    print(f"  DONE Full-LOO seed={seed} | F1={test_eval['f1_macro']:.4f} | PR={test_eval['pr_auc']:.4f} | FPR={100*test_eval['fp_rate']:.4f}%")
    del model,val,test
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return result

In [9]:
# Cell 9 — Run Step 7

def run_dataset_step7(dataset_name: str):
    print('\n'+'='*110+f'\nSTEP 7 DATASET: {dataset_name}\n'+'='*110)
    preflight_step5(dataset_name)
    bundle=load_fixed_subset_bundle(dataset_name)
    train_idx,val_idx,test_idx,scaler,split_meta=load_frozen_step5_artifacts(bundle)

    manifest={'dataset':dataset_name,'step7_tag':STEP7_EXPERIMENT_TAG,'step5_root':str(step5_dataset_dir(dataset_name)),
              'subset_fingerprint':bundle.subset_fingerprint,'split_fingerprint':split_meta['split_fingerprint'],
              'model_seeds':MODEL_SEEDS,'features':bundle.feature_names,'lightgbm_config':LIGHTGBM_CONFIG,
              'loo_config':{'folds':LOO_FOLDS,'seed':LOO_FOLD_SEED,'zero_unseen_nodes':LOO_ZERO_UNSEEN_NODES},
              'interpretation':'Full-LOO is a conservative cross-fitted no-self diagnostic; it reuses Step-5 Full checkpoints and does not retrain.'}
    atomic_json_dump(manifest,OUTPUT_ROOT/dataset_name/'experiment_manifest.json')

    val_windows=test_windows=None
    if RUN_FULL_LOO:
        val_windows=prepare_loo_windows(bundle,val_idx,scaler,'validation LOO')
        test_windows=prepare_loo_windows(bundle,test_idx,scaler,'test LOO')

    for seed in MODEL_SEEDS:
        print(f'\n--- seed {seed} ---')
        if RUN_LIGHTGBM: run_lightgbm_task(bundle,train_idx,val_idx,test_idx,scaler,split_meta,seed)
        if RUN_FULL_LOO: run_full_loo_task(bundle,val_windows,test_windows,split_meta,seed)

    del bundle,scaler,val_windows,test_windows
    gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

for dataset_name in DATASETS_TO_RUN:
    run_dataset_step7(dataset_name)
print('\nStep 7 tasks completed or resumed.')


STEP 7 DATASET: CIC_V2
Preflight OK: CIC_V2

Loading frozen subset CIC_V2: 496,384 rows, 39 features
  chunks=10 | selected=131,692/496,384
  chunks=20 | selected=263,031/496,384
  chunks=30 | selected=394,663/496,384
  chunks=38 | selected=496,384/496,384
Loaded in 1.6 min | label counts: {np.int8(0): np.int64(349993), np.int8(1): np.int64(146391)}
Frozen split: train=361,202, val=40,519, test=94,663 | 58cfcf6084a427575c09
  validation LOO: 9/9
Prepared 9 validation LOO windows in 0.0 min
  test LOO: 19/19
Prepared 19 test LOO windows in 0.0 min

--- seed 42 ---
[50]	valid_0's binary_logloss: 0.164023
[100]	valid_0's binary_logloss: 0.101864
[150]	valid_0's binary_logloss: 0.0892992
[200]	valid_0's binary_logloss: 0.0861406
[250]	valid_0's binary_logloss: 0.0852251
[300]	valid_0's binary_logloss: 0.0846293
[350]	valid_0's binary_logloss: 0.0844018
[400]	valid_0's binary_logloss: 0.0842001
[450]	valid_0's binary_logloss: 0.0841565
[500]	valid_0's binary_logloss: 0.083996
[550]	valid_0

In [10]:
# Cell 10 — Aggregate Step-5 Full + Step-7 results and create paper-ready tables

def normalize_result(result,source):
    test=result['test']
    return {'dataset':result['dataset'],'model':result.get('model',result.get('variant')),'seed':int(result['seed']),
            'f1_macro':float(test['f1_macro']),'auc_roc':float(test['auc_roc']),'pr_auc':float(test['pr_auc']),
            'fp_rate':float(test['fp_rate']),'tp_rate':float(test['tp_rate']),
            'tn':int(test['tn']),'fp':int(test['fp']),'fn':int(test['fn']),'tp':int(test['tp']),'source':source}

rows=[]; attack_rows=[]; completeness=[]
for dataset in DATA_PATH_CANDIDATES:
    for seed in MODEL_SEEDS:
        sources=[('full',step5_full_task_dir(dataset,seed)/'test_metrics.json','step5'),
                 ('lightgbm',step7_task_dir(dataset,'lightgbm',seed)/'test_metrics.json','step7'),
                 ('full_loo_crossfit',step7_task_dir(dataset,'full_loo_crossfit',seed)/'test_metrics.json','step7')]
        for model,path,stage in sources:
            complete=path.exists(); completeness.append({'dataset':dataset,'model':model,'seed':seed,'complete':complete,'path':str(path)})
            if not complete: continue
            result=load_json(path); row=normalize_result(result,stage); row['model']=model; rows.append(row)
            for a in result.get('per_attack',[]):
                attack_rows.append({'dataset':dataset,'model':model,'seed':seed,'attack':a['attack'],'support':a['count'],'recall':a['recall']})

runs=pd.DataFrame(rows); completeness_df=pd.DataFrame(completeness)
display(completeness_df)
if runs.empty: raise RuntimeError('No results found.')
if STRICT_COMPLETENESS and not completeness_df.complete.all(): raise RuntimeError('Step 7 incomplete.')
runs.to_csv(OUTPUT_ROOT/'step7_all_runs.csv',index=False)
completeness_df.to_csv(OUTPUT_ROOT/'step7_completeness.csv',index=False)

summary=[]
for (dataset,model),g in runs.groupby(['dataset','model']):
    r={'dataset':dataset,'model':model,'n_seeds':g.seed.nunique()}
    for m in ['f1_macro','auc_roc','pr_auc','fp_rate','tp_rate']:
        r[m+'_mean']=g[m].mean(); r[m+'_std']=g[m].std(ddof=1) if len(g)>1 else 0.0
    summary.append(r)
summary_df=pd.DataFrame(summary); summary_df.to_csv(OUTPUT_ROOT/'step7_model_summary.csv',index=False)

# Paired differences relative to the original Step-5 Full, same seeds.
paired=[]
for dataset in runs.dataset.unique():
    full=runs[(runs.dataset==dataset)&(runs.model=='full')]
    for other in ['lightgbm','full_loo_crossfit']:
        o=runs[(runs.dataset==dataset)&(runs.model==other)]
        for _,r in full.merge(o,on=['dataset','seed'],suffixes=('_full','_other')).iterrows():
            paired.append({'dataset':dataset,'comparison':f'full_minus_{other}','seed':r.seed,
                           'delta_f1_macro':r.f1_macro_full-r.f1_macro_other,
                           'delta_pr_auc':r.pr_auc_full-r.pr_auc_other,
                           'delta_fp_rate':r.fp_rate_full-r.fp_rate_other,
                           'delta_tp_rate':r.tp_rate_full-r.tp_rate_other})
paired_df=pd.DataFrame(paired); paired_df.to_csv(OUTPUT_ROOT/'step7_paired_differences.csv',index=False)

if attack_rows:
    pd.DataFrame(attack_rows).to_csv(OUTPUT_ROOT/'step7_per_attack_all_runs.csv',index=False)

# Paper-ready rows: FPR is explicitly converted from rate to percentage.
def pm(mean,std,d=4): return f'{mean:.{d}f}±{std:.{d}f}'
def fpr_pm(mean,std):
    a,b=100*mean,100*std; d=4 if a<0.1 else 3
    return f'{a:.{d}f}±{b:.{d}f}'

paper=[]
for _,r in summary_df.iterrows():
    paper.append({'dataset':r.dataset,'method':{'full':'Full','lightgbm':'LightGBM','full_loo_crossfit':'Full-LOO'}.get(r.model,r.model),
                  'F1-macro':pm(r.f1_macro_mean,r.f1_macro_std),
                  'PR-AUC':pm(r.pr_auc_mean,r.pr_auc_std),
                  'FPR (%)':fpr_pm(r.fp_rate_mean,r.fp_rate_std),
                  'TPR':f'{r.tp_rate_mean:.4f}'})
paper_df=pd.DataFrame(paper); paper_df.to_csv(OUTPUT_ROOT/'step7_paper_fill_rows.csv',index=False)
print('\nPaper fill rows'); display(paper_df.sort_values(['dataset','method']))
print('\nPaired differences'); display(paired_df)

,dataset,model,seed,complete,path
0,UNSW_V2,full,42,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
1,UNSW_V2,lightgbm,42,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
2,UNSW_V2,full_loo_crossfit,42,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
3,UNSW_V2,full,123,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
4,UNSW_V2,lightgbm,123,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
5,UNSW_V2,full_loo_crossfit,123,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
6,UNSW_V2,full,456,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
7,UNSW_V2,lightgbm,456,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
8,UNSW_V2,full_loo_crossfit,456,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...
9,CIC_V2,full,42,True,/content/drive/MyDrive/THAT_IDS_RERUN/conferen...



Paper fill rows


,dataset,method,F1-macro,PR-AUC,FPR (%),TPR
0,CIC_V2,Full,0.9736±0.0011,0.9840±0.0017,0.0223±0.0056,0.9226
1,CIC_V2,Full-LOO,0.9737±0.0007,0.9825±0.0014,0.0144±0.0049,0.9226
2,CIC_V2,LightGBM,0.9786±0.0001,0.9747±0.0001,0.0441±0.0103,0.9377
3,UNSW_V2,Full,0.9948±0.0019,0.9974±0.0002,0.606±0.221,0.9991
4,UNSW_V2,Full-LOO,0.9944±0.0018,0.9975±0.0002,0.631±0.209,0.9985
5,UNSW_V2,LightGBM,0.9964±0.0001,0.9991±0.0000,0.429±0.000,0.9997



Paired differences


,dataset,comparison,seed,delta_f1_macro,delta_pr_auc,delta_fp_rate,delta_tp_rate
0,UNSW_V2,full_minus_lightgbm,42,-0.000461,-0.001500,0.000604,0.000068
1,UNSW_V2,full_minus_lightgbm,123,-0.003905,-0.001900,0.004317,-0.001215
2,UNSW_V2,full_minus_lightgbm,456,-0.000487,-0.001598,0.000392,-0.000473
3,UNSW_V2,full_minus_full_loo_crossfit,42,0.000463,-0.000084,-0.000302,0.000608
4,UNSW_V2,full_minus_full_loo_crossfit,123,0.000292,-0.000087,-0.000121,0.000540
5,UNSW_V2,full_minus_full_loo_crossfit,456,0.000536,-0.000085,-0.000332,0.000743
6,CIC_V2,full_minus_lightgbm,42,-0.006208,0.008083,-0.000334,-0.018922
7,CIC_V2,full_minus_lightgbm,123,-0.004163,0.008661,-0.000125,-0.012424
8,CIC_V2,full_minus_lightgbm,456,-0.004611,0.011117,-0.000195,-0.013917
9,CIC_V2,full_minus_full_loo_crossfit,42,-0.000574,0.001260,0.000125,-0.001273


In [11]:
# Cell 11 — Generate manuscript fill guide, operating points, and compact result bundle

# Test-curve operating points for the two new Step-7 methods.
curve=[]
for dataset in DATA_PATH_CANDIDATES:
    for model in ['lightgbm','full_loo_crossfit']:
        for seed in MODEL_SEEDS:
            p=step7_task_dir(dataset,model,seed)/'test_predictions.npz'
            if not p.exists(): continue
            d=np.load(p,allow_pickle=True)
            for point in test_curve_operating_points(d['scores'],d['labels']):
                curve.append({'dataset':dataset,'model':model,'seed':seed,**point})
curve_df=pd.DataFrame(curve); curve_df.to_csv(OUTPUT_ROOT/'step7_test_curve_operating_points.csv',index=False)

lines=['# Step 7 manuscript fill guide','',
       'Use values exactly as shown in `step7_paper_fill_rows.csv`. FPR values are percentages, not rates.','']
for dataset,label in [('UNSW_V2','Table VII — NF-UNSW-NB15-v2'),('CIC_V2','Table VIII — NF-CSE-CIC-IDS2018-v2')]:
    lines += [f'## {label}']
    for method in ['LightGBM','Full-LOO']:
        q=paper_df[(paper_df.dataset==dataset)&(paper_df.method==method)]
        if q.empty:
            lines.append(f'- {method}: **TBD — Step 7 incomplete**')
        else:
            r=q.iloc[0]
            lines.append(f"- {method}: F1={r['F1-macro']}; PR-AUC={r['PR-AUC']}; FPR={r['FPR (%)']}%; TPR={r['TPR']}.")
    lines.append('')

# Generate cautious result language only when all three seed summaries exist.
for dataset in ['UNSW_V2','CIC_V2']:
    s=summary_df[summary_df.dataset==dataset].set_index('model')
    if {'full','lightgbm','full_loo_crossfit'}.issubset(s.index):
        full=s.loc['full']; lgbm=s.loc['lightgbm']; loo=s.loc['full_loo_crossfit']
        lines += [f'## Suggested prose — {dataset}',
                  f"- Full vs LightGBM: ΔF1={full.f1_macro_mean-lgbm.f1_macro_mean:+.6f}, ΔPR-AUC={full.pr_auc_mean-lgbm.pr_auc_mean:+.6f}, ΔFPR={100*(full.fp_rate_mean-lgbm.fp_rate_mean):+.4f} percentage points.",
                  f"- Full vs Full-LOO: ΔF1={full.f1_macro_mean-loo.f1_macro_mean:+.6f}, ΔPR-AUC={full.pr_auc_mean-loo.pr_auc_mean:+.6f}, ΔFPR={100*(full.fp_rate_mean-loo.fp_rate_mean):+.4f} percentage points.",
                  '- Interpret Full-LOO as a conservative no-self diagnostic, not an exact one-at-a-time retraining experiment.','']

guide=OUTPUT_ROOT/'step7_manuscript_fill_guide.md'; guide.write_text('\n'.join(lines),encoding='utf-8')

bundle=OUTPUT_ROOT/'THAT_IDS_step7_results_bundle.zip'
small=['step7_all_runs.csv','step7_completeness.csv','step7_model_summary.csv','step7_paired_differences.csv',
       'step7_per_attack_all_runs.csv','step7_paper_fill_rows.csv','step7_test_curve_operating_points.csv','step7_manuscript_fill_guide.md']
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for name in small:
        p=OUTPUT_ROOT/name
        if p.exists(): z.write(p,arcname=name)
    for dataset in DATA_PATH_CANDIDATES:
        p=OUTPUT_ROOT/dataset/'experiment_manifest.json'
        if p.exists(): z.write(p,arcname=f'manifests/{dataset}.json')
        for model in ['lightgbm','full_loo_crossfit']:
            for seed in MODEL_SEEDS:
                d=step7_task_dir(dataset,model,seed)
                for fn in ['test_metrics.json','completed.json','feature_importance.csv']:
                    p=d/fn
                    if p.exists(): z.write(p,arcname=f'tasks/{dataset}/{model}/seed_{seed}/{fn}')

print('Fill guide:',guide)
print('Result bundle:',bundle)
print(guide.read_text(encoding='utf-8')[:5000])

Fill guide: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step7_tabular_loo_v1/step7_manuscript_fill_guide.md
Result bundle: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step7_tabular_loo_v1/THAT_IDS_step7_results_bundle.zip
# Step 7 manuscript fill guide

Use values exactly as shown in `step7_paper_fill_rows.csv`. FPR values are percentages, not rates.

## Table VII — NF-UNSW-NB15-v2
- LightGBM: F1=0.9964±0.0001; PR-AUC=0.9991±0.0000; FPR=0.429±0.000%; TPR=0.9997.
- Full-LOO: F1=0.9944±0.0018; PR-AUC=0.9975±0.0002; FPR=0.631±0.209%; TPR=0.9985.

## Table VIII — NF-CSE-CIC-IDS2018-v2
- LightGBM: F1=0.9786±0.0001; PR-AUC=0.9747±0.0001; FPR=0.0441±0.0103%; TPR=0.9377.
- Full-LOO: F1=0.9737±0.0007; PR-AUC=0.9825±0.0014; FPR=0.0144±0.0049%; TPR=0.9226.

## Suggested prose — UNSW_V2
- Full vs LightGBM: ΔF1=-0.001618, ΔPR-AUC=-0.001666, ΔFPR=+0.1771 percentage points.
- Full vs Full-LOO: ΔF1=+0.000430, ΔPR-AUC=-0.000086, ΔFPR=-0.0252 percentage points.
- Interpret Full-LOO as a conser